In [1]:
import numpy as np                                                                                                                                          
import pandas as pd                                                                                                                                         
import matplotlib.pyplot as plt                                                                                                                             
from matplotlib.pyplot import figure                                                                                                                        
import random                                                                                                                                               
import re                                                                                                                                                   

pd.set_option('display.max_rows', 53)                                                                                                                       
pd.set_option('display.max_columns', 250)    

Generates the plots of Figures 2l and Table S2

# Functions

In [2]:
def uniqueElements(arr, n):
    # First sort the array so that 
    # all occurrences become consecutive
    arr.sort()

    # Create a list to store unique elements
    unique_elements = []

    # Traverse the sorted array
    i = 0
    while i < n:
        # Move the index ahead while there are duplicates
        if i < n-1 and arr[i] == arr[i+1]:
            while i < n-1 and arr[i] == arr[i+1]:
                i += 1
        # Add the last occurrence of the current element to the unique list
        unique_elements.append(arr[i])
        i += 1

    return unique_elements

In [3]:
def getBinNb(AF):
    binNb = 0
    for i in range(100):
        if (i*0.01 < AF) and (AF <= (i+1)*0.01):
            binNb = i+1
    return binNb

In [4]:
def nthPand(df_ref,n_divide):
        
    n = random.randint(1, n_divide)
        
    num_rows = len(df_ref)

    rows_per_chunk = num_rows // n_divide

    nth_part = df_ref.iloc[(n-1)*rows_per_chunk : n*rows_per_chunk]
    
    return nth_part

In [5]:
def MAF_Distrib_list(listAFinput,x,lowerThreshold,col,maxY):   
    
    listAF = listAFinput.copy()
#     for i in range(len(listAF)):
#         if listAF[i]>0.5:
#             listAF[i]=1-listAF[i]   
            
    NbPerBags=[0]*len(x)
    
    for i in listAF:    
        for j in range(0,len(x)):
        
            if j==0:
                if lowerThreshold<i and i<=x[0]:
                    NbPerBags[j]+=1
        
            else:
                if x[j-1]<i and i<=x[j]:
                    NbPerBags[j]+=1
                            
    print("total number of points is "+ str(sum(NbPerBags))) 
    total=sum(NbPerBags)
    for i in range(len(NbPerBags)):
        NbPerBags[i]=NbPerBags[i]/total*100
    
    figure(figsize=(4, 3), dpi=120)
    
    x_str=[]
    for i in range(len(x)):
        if i == 0:
            x_str.append(str(lowerThreshold)+'- '+str(x[i]))
        else:
            x_str.append(str(x[i-1]) + ' - '+str(x[i]))
    
    #print(NbPerBags)
    plt.ylabel("% of variants  ",fontsize='10')  
    plt.xticks(fontsize='8',rotation=45)
    plt.yticks(fontsize='8')
    #plt.title("Disitribution of variants' MAF in "+ pop +' population with MAF above ' + str(lowerThreshold))
    plt.bar(x_str, NbPerBags, color=col)
    
    plt.ylim(0, maxY)    
    
    plt.show()

In [6]:
def subPanda_popSpe(var_sign,var_notSign, timesSize):
    
    populations = ['afr','amr','nfe','sas','fin','asj','eas']
    dfs = []
    
    for pop in populations:
                
        var_sign_pop = var_sign[var_sign['pop']==pop].reset_index(drop=True)
        var_notSign_pop = var_notSign[var_notSign['pop']==pop].reset_index(drop=True)
        af_col = f"AF_{pop.lower()}" 
              
        spd_HWE_pop = subPanda(var_sign_pop[af_col].tolist(),var_notSign_pop,timesSize,af_col)      
        
        dfs.append(spd_HWE_pop)
        
    return pd.concat(dfs, ignore_index=True)
    

In [7]:
def subPanda(AF_list,df_ref,timesSize,colName):
    
    df = df_ref.copy()
    
#     print(len(df))
    
    indices = []
    
    rr=0
    
#     print(len(AF_list))
    for i in range(timesSize):
        
#         print(i)
        
        for af in AF_list:
            
#             print(af)


    #         if rr%500==0:
    #             print(str(rr) + ' lines read at ' + str(datetime.now()))

            binNb = getBinNb(af)
            low = 0.01 * (binNb -1)
            up = 0.01 * binNb

            n_divide = 100

            nth_part = nthPand(df,n_divide)

            df_temp = nth_part[(low < nth_part[colName]) & (nth_part[colName] <= up)]

            while len(df_temp) < 1:
                nth_part = nthPand(df,n_divide)
                df_temp = nth_part[(low < nth_part[colName]) & (nth_part[colName] <= up)]

            df_temp_sample = df_temp.sample(n=1)

            indices.append(df_temp_sample.index[0])

            rr+=1
#     print(len(indices))
#     print(max(indices))
        
    return df.iloc[indices]
    
    

In [8]:
import numpy as np
from scipy import stats
import pandas as pd
import statsmodels.stats.contingency_tables as ct

def fisherTest(contingency_table):
    print(contingency_table)

    odds_ratio, p_value = stats.fisher_exact(contingency_table)

    ci_lower, ci_upper = ct.Table2x2(contingency_table).oddsratio_confint(method='exact')

    nb_HWD_In = contingency_table[0][0]
    nb_HWE_In = contingency_table[1][0]
    nb_HWD = np.sum(contingency_table[0])
    nb_HWE = np.sum(contingency_table[1])

    print(f"\n% of hits among HWD variants: {round(nb_HWD_In / nb_HWD * 100, 3)}%")
    print(f"% of hits among HWE variants: {round(nb_HWE_In / nb_HWE * 100, 3)}%")
    print(f"\nOdds Ratio: {odds_ratio}")
    print(f'95% CI: [{ci_lower:.3f}, {ci_upper:.3f}]')
    print(f"P-value: {p_value:.3e}\n")

    return odds_ratio, ci_lower, ci_upper, p_value


In [9]:
def fisherTable(var_HWE, var_HWD, variants_df):
    # --- Clean GWAS df to only keep rows with valid numeric CHR_IDs ---
    valid_chroms = set(map(str, range(1, 23)))

    def clean_chr(chr_val):
        try:
            val = str(int(float(chr_val)))  # Converts "1.0" → 1 → "1"
            return val if val in valid_chroms else None
        except:
            return None  # Exclude non-numeric values like "NR", "X", "Y", etc.

    variants_df = variants_df.copy()
    variants_df["clean_chr"] = variants_df["CHR_ID"].apply(clean_chr)

    # Remove invalid rows
    variants_df = variants_df[variants_df["clean_chr"].notnull() & variants_df["CHR_POS"].notnull()]

    # Build the set of "chr:pos" strings
    gwas_set = set(
        "chr" + row.clean_chr + ":" + str(int(row.CHR_POS))
        for _, row in variants_df.iterrows()
    )
    
#     print(len(gwas_set))

    # Add "chr:pos" to each variant set
    var_HWE = var_HWE.copy()
    var_HWD = var_HWD.copy()
    var_HWE["loc"] = var_HWE["chromo"].astype(str).str.strip() + ":" + var_HWE["position"].astype(int).astype(str)
    var_HWD["loc"] = var_HWD["chromo"].astype(str).str.strip() + ":" + var_HWD["position"].astype(int).astype(str)

    # Subset
    var_HWE_In = var_HWE[var_HWE["loc"].isin(gwas_set)]
    var_HWD_In = var_HWD[var_HWD["loc"].isin(gwas_set)]

    nb_HWE_In = len(var_HWE_In)
    nb_HWD_In = len(var_HWD_In)
    nb_HWE_Out = len(var_HWE) - nb_HWE_In
    nb_HWD_Out = len(var_HWD) - nb_HWD_In

    contingency_table = np.array([
        [nb_HWD_In, nb_HWD_Out],
        [nb_HWE_In, nb_HWE_Out]
    ])

    fisherTest(contingency_table)
    
    return contingency_table

In [10]:
def fisherTable_HGMD(var_HWE, var_HWD, variants_df):

    import re

    def extract_allele_change(hgvs_str):
        """Extract the change suffix from a single HGVS-like entry (no semicolons).
        Strips transcript prefix up to and including 'c.' or 'n.', then returns
        everything after the last digit of the positional part.

        e.g. '964-1G>C'                    -> '-1G>C'
             'ENST00000355527.8:c.964-1G>C' -> '-1G>C'
             '100G>C'                       -> 'G>C'
             '100delA'                      -> 'delA'
             '100_102del'                   -> '_102del'
        """
        s = str(hgvs_str).strip()
        # Strip transcript prefix (up to and including 'c.' or 'n.')
        for prefix in ['c.', 'n.']:
            if prefix in s:
                s = s.split(prefix)[-1]
                break
        m = re.search(r'\d([^0-9].*)$', s)
        return m.group(1) if m else None

    def extract_all_allele_changes(hgvs_str):
        """Extract allele changes from all transcripts in a (possibly multi-transcript)
        HGVSc string. Returns a set of allele change strings.
        Used for allVar HGVSc which can contain multiple transcripts separated by ';'.
        Handles Case 3 (overlapping genes on opposite strands) by trying all transcripts.
        """
        if pd.isna(hgvs_str):
            return set()
        entries = [e.strip() for e in str(hgvs_str).strip(';').split(';') if e.strip()]
        changes = set()
        for entry in entries:
            ch = extract_allele_change(entry)
            if ch:
                changes.add(ch)
        return changes

    valid_chroms = set(map(str, range(1, 23)))

    def clean_chr(chr_val):
        try:
            val = str(int(float(chr_val)))
            return val if val in valid_chroms else None
        except:
            return None

    variants_df = variants_df.copy()
    variants_df["clean_chr"] = variants_df["chromosome"].apply(clean_chr)
    variants_df = variants_df[variants_df["clean_chr"].notnull() & variants_df["startCoord"].notnull()]

    # Extract allele change from HGMD hgvs column (always a single entry)
    variants_df["allele_change"] = variants_df["hgvs"].apply(extract_allele_change)

    # Build set of "chr:pos:allele_change" strings
    hgmd_set = set(
        "chr" + row.clean_chr + ":" + str(int(row.startCoord)) + ":" + row.allele_change
        for _, row in variants_df.iterrows()
        if pd.notna(row.allele_change)
    )

    var_HWE = var_HWE.copy()
    var_HWD = var_HWD.copy()

    # Build all possible match keys per allVar variant (one per transcript)
    # A variant matches HGMD if any of its transcript allele changes hits hgmd_set
    def make_keys(row):
        loc = str(row["chromo"]).strip() + ":" + str(int(row["position"]))
        changes = extract_all_allele_changes(row["HGVSc"])
        return {loc + ":" + ch for ch in changes}

    var_HWE["match_keys"] = var_HWE.apply(make_keys, axis=1)
    var_HWD["match_keys"] = var_HWD.apply(make_keys, axis=1)

    var_HWE["in_hgmd"] = var_HWE["match_keys"].apply(lambda keys: bool(keys & hgmd_set))
    var_HWD["in_hgmd"] = var_HWD["match_keys"].apply(lambda keys: bool(keys & hgmd_set))

    var_HWE_In = var_HWE[var_HWE["in_hgmd"]]
    var_HWD_In = var_HWD[var_HWD["in_hgmd"]]

    nb_HWE_In = len(var_HWE_In)
    nb_HWD_In = len(var_HWD_In)
    nb_HWE_Out = len(var_HWE) - nb_HWE_In
    nb_HWD_Out = len(var_HWD) - nb_HWD_In

    contingency_table = np.array([
        [nb_HWD_In, nb_HWD_Out],
        [nb_HWE_In, nb_HWE_Out]
    ])

#     print(var_HWD_In["vid_v4pop"].tolist())
    fisherTest(contingency_table)
    return contingency_table


In [11]:
def fisherTable_ClinVar(var_HWE, var_HWD, variants_df):

    valid_chroms = set(map(str, range(1, 23)))

    def clean_chr(chr_val):
        try:
            val = str(int(float(chr_val)))
            return val if val in valid_chroms else None
        except:
            return None

    variants_df = variants_df.copy()
    variants_df["clean_chr"] = variants_df["Chromosome"].apply(clean_chr)
    variants_df = variants_df[
        variants_df["clean_chr"].notnull() &
        variants_df["Start"].notnull() &
        (variants_df["ReferenceAlleleVCF"] != "na") &
        (variants_df["AlternateAlleleVCF"] != "na")
    ]

    # Build set of "chr:pos:ref:alt" strings
    # Use Start (not PositionVCF) as it is updated to hg38 during liftover for GRCh37 entries
    clinvar_set = set(
        "chr" + row.clean_chr + ":" + str(int(row.Start)) + ":" +
        str(row.ReferenceAlleleVCF) + ":" + str(row.AlternateAlleleVCF)
        for _, row in variants_df.iterrows()
    )

    var_HWE = var_HWE.copy()
    var_HWD = var_HWD.copy()

    # Build chr:pos:ref:alt key from allVar columns
    var_HWE["match_key"] = (var_HWE["chromo"].astype(str).str.strip() + ":" +
                             var_HWE["position"].astype(int).astype(str) + ":" +
                             var_HWE["ref"].astype(str) + ":" +
                             var_HWE["alt"].astype(str))
    var_HWD["match_key"] = (var_HWD["chromo"].astype(str).str.strip() + ":" +
                             var_HWD["position"].astype(int).astype(str) + ":" +
                             var_HWD["ref"].astype(str) + ":" +
                             var_HWD["alt"].astype(str))

    var_HWE_In = var_HWE[var_HWE["match_key"].isin(clinvar_set)]
    var_HWD_In = var_HWD[var_HWD["match_key"].isin(clinvar_set)]

    nb_HWE_In = len(var_HWE_In)
    nb_HWD_In = len(var_HWD_In)
    nb_HWE_Out = len(var_HWE) - nb_HWE_In
    nb_HWD_Out = len(var_HWD) - nb_HWD_In

    contingency_table = np.array([
        [nb_HWD_In, nb_HWD_Out],
        [nb_HWE_In, nb_HWE_Out]
    ])

#     print(var_HWD_In["vid_v4pop"].tolist())
    fisherTest(contingency_table)
    return contingency_table


In [12]:
def fisherTable_cisQTL(var_HWE, var_HWD, variants_df):
    # --- Clean GWAS df to only keep rows with valid numeric CHR_IDs ---
    valid_chroms = set(map(str, range(1, 23)))

    def clean_chr(chr_val):
        try:
            val = str(int(float(chr_val)))  # Converts "1.0" → 1 → "1"
            return val if val in valid_chroms else None
        except:
            return None  # Exclude non-numeric values like "NR", "X", "Y", etc.

    variants_df = variants_df.copy()
    variants_df["clean_chr"] = variants_df["SNPChr"].apply(clean_chr)

    # Remove invalid rows
    variants_df = variants_df[variants_df["clean_chr"].notnull() & variants_df["SNPPos"].notnull()]
    
#     print(len(variants_df))

    # Build the set of "chr:pos" strings
    gwas_set = set(
        "chr" + row.clean_chr + ":" + str(int(row.SNPPos))
        for _, row in variants_df.iterrows()
    )
    
#     print(len(gwas_set))

    # Add "chr:pos" to each variant set
    var_HWE = var_HWE.copy()
    var_HWD = var_HWD.copy()
    var_HWE["loc"] = var_HWE["chromo"].astype(str).str.strip() + ":" + var_HWE["position"].astype(int).astype(str)
    var_HWD["loc"] = var_HWD["chromo"].astype(str).str.strip() + ":" + var_HWD["position"].astype(int).astype(str)

    # Subset
    var_HWE_In = var_HWE[var_HWE["loc"].isin(gwas_set)]
    var_HWD_In = var_HWD[var_HWD["loc"].isin(gwas_set)]

    nb_HWE_In = len(var_HWE_In)
    nb_HWD_In = len(var_HWD_In)
    nb_HWE_Out = len(var_HWE) - nb_HWE_In
    nb_HWD_Out = len(var_HWD) - nb_HWD_In

    contingency_table = np.array([
        [nb_HWD_In, nb_HWD_Out],
        [nb_HWE_In, nb_HWE_Out]
    ])
    
#     print(var_HWD_In['vid_v4pop'].tolist())

    fisherTest(contingency_table)
    
    return contingency_table


In [13]:
def fisherTable_GPN_MSA(var_HWE, var_HWD):
    """
    Enrichment of deep negative scores (< -5 and < -7)
    between HWD and HWE sampled sets.
    """

    listCTs = []
    
    thresholds = [-5, -7]

    for thr in thresholds:
        print(f"\n===== Score threshold: < {thr} =====")

        # Counts for HWD
        HWD_in  = (var_HWD["score"] < thr).sum()
        HWD_out = len(var_HWD) - HWD_in

        # Counts for HWE
        HWE_in  = (var_HWE["score"] < thr).sum()
        HWE_out = len(var_HWE) - HWE_in

        contingency_table = np.array([
            [HWD_in, HWD_out],   # row 0 = HWD
            [HWE_in, HWE_out]    # row 1 = HWE
        ])
        
#         print(var_HWD[var_HWD["score"] < thr]['vid_v4pop'].tolist())

        fisherTest(contingency_table)
    
        listCTs.append(contingency_table)
    
    return listCTs


In [14]:
def fisherTable_Impact(var_HWE, var_HWD):
    impacts = ['MODERATE', 'MODIFIER', 'LOW', 'HIGH']
    
    listCTs = []

    for impact in impacts:
        print(f"\n===== Impact category: {impact} =====")

        # Count variants in each category
        # (A variant is counted as "in" if the impact appears anywhere in its list)
        HWD_in  = var_HWD['impact'].str.contains(impact).sum()
        HWD_out = len(var_HWD) - HWD_in

        HWE_in  = var_HWE['impact'].str.contains(impact).sum()
        HWE_out = len(var_HWE) - HWE_in

        contingency_table = np.array([
            [HWD_in, HWD_out],   # Row 0 = HWD
            [HWE_in, HWE_out]    # Row 1 = HWE
        ])
        
#         print(var_HWD[var_HWD['impact'].str.contains(impact)]['vid_v4pop'].tolist())

        fisherTest(contingency_table)
    
        listCTs.append(contingency_table)
    
    return listCTs


In [15]:
def fisherTable_Consequence(var_HWE, var_HWD):
    
    consequences = ['missense_variant',
     'downstream_gene_variant',
     'synonymous_variant',
     'splice_donor_region_variant',
     'intron_variant',
     'upstream_gene_variant',
     'splice_polypyrimidine_tract_variant',
     'splice_region_variant',
     '5_prime_UTR_variant',
     '3_prime_UTR_variant',
     'inframe_insertion',
     'splice_donor_5th_base_variant',
     'stop_lost',
     'inframe_deletion',
     'frameshift_variant',
     'stop_gained',
     'non_coding_transcript_exon_variant',
     'regulatory_region_variant',
     'TF_binding_site_variant',
     'stop_retained_variant',
     'NMD_transcript_variant',
     'non_coding_transcript_variant',
     'splice_donor_variant',
     'start_lost',
     'splice_acceptor_variant',
     'protein_altering_variant',
     'coding_sequence_variant',
     'start_retained_variant']
    
    listCTs = []

    for conseq in consequences:
        print(f"\n===== Impact category: {conseq} =====")

        # Count variants in each category
        # (A variant is counted as "in" if the impact appears anywhere in its list)
        HWD_in  = var_HWD['consequence'].str.contains(conseq).sum()
        HWD_out = len(var_HWD) - HWD_in

        HWE_in  = var_HWE['consequence'].str.contains(conseq).sum()
        HWE_out = len(var_HWE) - HWE_in

        contingency_table = np.array([
            [HWD_in, HWD_out],   # Row 0 = HWD
            [HWE_in, HWE_out]    # Row 1 = HWE
        ])
        
#         print(var_HWD[var_HWD['consequence'].str.contains(conseq)]['vid_v4pop'].tolist())

        fisherTest(contingency_table)
    
        listCTs.append(contingency_table)
    
    return listCTs


In [16]:
def sampleEnrich_HGMD(HWD,HWE,variants_df,timesSize,):
    
    print('-------------------------------------HGMD----------------------------------------------------')

    
    spd_HWE = subPanda_popSpe(HWD,HWE,timesSize)
    contingency_table = fisherTable_HGMD(spd_HWE,HWD,variants_df)
    
    return contingency_table
    
    
#     print(len(HWD[HWD['ethn_v4']=='NFE'])/len(HWD))
#     print(len(spd_HWE[spd_HWE['ethn_v4']=='NFE'])/len(spd_HWE))
    
#     intervals = [0.01,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
    
#     MAF_Distrib_list(HWD[colName],intervals,0,'blue',100)
#     MAF_Distrib_list(spd_HWE[colName],intervals,0,'blue',100)

In [17]:
def sampleEnrich_ClinVar(HWD,HWE,variants_df,timesSize,):
    
    print('-------------------------------------ClinVar----------------------------------------------------')
    
    spd_HWE = subPanda_popSpe(HWD,HWE,timesSize)
    contingency_table = fisherTable_ClinVar(spd_HWE,HWD,variants_df)
    
    return contingency_table
    
#     print(len(HWD[HWD['ethn_v4']=='AMR'])/len(HWD))
#     print(len(spd_HWE[spd_HWE['ethn_v4']=='AMR'])/len(spd_HWE))
    
#     intervals = [0.01,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
    
#     MAF_Distrib_list(HWD[colName],intervals,0,'blue',100)
#     MAF_Distrib_list(spd_HWE[colName],intervals,0,'blue',100)

In [18]:
def sampleEnrich_GWAS(HWD,HWE,variants_df,timesSize,):
    
    print('-------------------------------------GWAS----------------------------------------------------')
    
    spd_HWE = subPanda_popSpe(HWD,HWE,timesSize)
    contingency_table = fisherTable(spd_HWE,HWD,variants_df)
    
    return contingency_table
    
#     intervals = [0.01,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
    
#     MAF_Distrib_list(HWD[colName],intervals,0,'blue',100)
#     MAF_Distrib_list(spd_HWE[colName],intervals,0,'blue',100)

In [19]:
def sampleEnrich_cisQTL(HWD,HWE,variants_df,timesSize,):
    
    print('-------------------------------------cis eQTL----------------------------------------------------')
    
    spd_HWE = subPanda_popSpe(HWD,HWE,timesSize)
    contingency_table = fisherTable_cisQTL(spd_HWE,HWD,variants_df)
    
    return contingency_table
    
#     intervals = [0.01,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
    
#     MAF_Distrib_list(HWD[colName],intervals,0,'blue',100)
#     MAF_Distrib_list(spd_HWE[colName],intervals,0,'blue',100)

In [20]:
def sampleEnrich_Impact(HWD,HWE,timesSize,):
    
    print('-------------------------------------Impact----------------------------------------------------')
    
    spd_HWE = subPanda_popSpe(HWD,HWE,timesSize)
    list_CTs = fisherTable_Impact(spd_HWE,HWD)
    
    return list_CTs
    
#     print(len(HWD[HWD['ethn_v4']=='AFR'])/len(HWD))
#     print(len(spd_HWE[spd_HWE['ethn_v4']=='AFR'])/len(spd_HWE))
    
#     intervals = [0.01,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
    
#     MAF_Distrib_list(HWD[colName],intervals,0,'blue',100)
#     MAF_Distrib_list(spd_HWE[colName],intervals,0,'blue',100)

In [21]:
def sampleEnrich_Consequence(HWD,HWE,timesSize,):
    
    print('-------------------------------------Consequence----------------------------------------------------')
    
    spd_HWE = subPanda_popSpe(HWD,HWE,timesSize)
    list_CTs = fisherTable_Consequence(spd_HWE,HWD)
    
    return list_CTs
    
#     print(len(HWD[HWD['ethn_v4']=='AMR'])/len(HWD))
#     print(len(spd_HWE[spd_HWE['ethn_v4']=='AMR'])/len(spd_HWE))
    
#     intervals = [0.01,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
    
#     MAF_Distrib_list(HWD[colName],intervals,0,'blue',100)
#     MAF_Distrib_list(spd_HWE[colName],intervals,0,'blue',100)

In [22]:
def sampleEnrich_GPN_MSA(HWD,HWE,timesSize,):
    
    print('-------------------------------------GPN - MSA----------------------------------------------------')
    
    spd_HWE = subPanda_popSpe(HWD,HWE,timesSize)
    list_CTs = fisherTable_GPN_MSA(spd_HWE,HWD)
    
    return list_CTs
    
#     print(len(HWD[HWD['ethn_v4']=='NFE'])/len(HWD))
#     print(len(spd_HWE[spd_HWE['ethn_v4']=='NFE'])/len(spd_HWE))
    
#     intervals = [0.01,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
    
#     MAF_Distrib_list(HWD[colName],intervals,0,'blue',100)
#     MAF_Distrib_list(spd_HWE[colName],intervals,0,'blue',100)

In [23]:
def split_significant_variants(
    full_tsv,
    tags_tsv,
    out_notsign
):

    # --- Load main RUTH table ---
    df = pd.read_csv(full_tsv, sep="\t")
#     print(len(df))


    # --- Load tag list ---
    tags_df = pd.read_csv(tags_tsv, sep="\t")
    tags = set(tags_df["vid_v4pop"])
    
#     print(len(tags_df))
#     print(len(tags))

    # --- Split ---
    notsign_df = df[~df["vid_v4pop"].isin(tags)].copy()
    
#     print(len(notsign_df))

    # --- Save ---
    notsign_df.to_csv(out_notsign, sep="\t", index=False)

    print(f"✅ {len(notsign_df)} non-tag variants saved to {out_notsign}")

    return notsign_df


# v4 gnomAD data

In [24]:
# allVar_v4 = pd.read_csv('data/v4_df_GT.tsv', sep='\t')
# HWD_v4 = pd.read_csv('data/v4_df_GT_BH.tsv', sep='\t')

# # Construct vid and vid_v4pop for HWD_v4 (not present in v4_df_GT_BH)
# HWD_v4['vid'] = HWD_v4['chromo'].astype(str) + ':' + HWD_v4['position'].astype(str) + ':' + HWD_v4['ref'].astype(str) + ':' + HWD_v4['alt'].astype(str)
# HWD_v4['vid_v4pop'] = HWD_v4['vid'].astype(str) + '_' + HWD_v4['pop'].astype(str)

# print('allVar_v4 shape:', allVar_v4.shape)
# print('HWD_v4 shape:', HWD_v4.shape)

In [25]:
# # Fix: add vid and vid_v4pop to v4_df_GT_allpop.tsv (was written before these columns were created)                                                                                                              
# allVar_v4 = pd.read_csv('../data/figures/v4_df_GT_allpop.tsv', sep='\t')                                                                                                                                                         
# allVar_v4['vid'] = allVar_v4['chromo'].astype(str) + ':' + allVar_v4['position'].astype(str) + ':' + allVar_v4['ref'].astype(str) + ':' + allVar_v4['alt'].astype(str)
# allVar_v4['vid_v4pop'] = allVar_v4['vid'].astype(str) + '_' + allVar_v4['pop'].astype(str)                                                                                                                       
# allVar_v4.to_csv('../data/figures/v4_df_GT_allpop.tsv', sep='\t', index=False)

In [26]:
# allVar_v4.to_csv('../data/figures/v4_df_GT_allpop.tsv', sep='\t', index=False)
# HWD_v4.to_csv('../data/figures/v4_df_GT_BH_allpop.tsv', sep='\t', index=False)

In [27]:
allVar_v4 = pd.read_csv('../data/figures/v4_df_GT_allpop.tsv', sep='\t')
HWD_v4 = pd.read_csv('../data/figures/v4_df_GT_BH_allpop.tsv', sep='\t')

/var/folders/q8/5rgt92sx7b74krhrt6l4_7hw0000gn/T/ipykernel_36662/2086606334.py:1: DtypeWarning: Columns (46,47,48,49,50,51,52,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,110,156,159,160,161,162,163,168,169,170,171) have mixed types. Specify dtype option on import or set low_memory=False.
  allVar_v4 = pd.read_csv('../data/figures/v4_df_GT_allpop.tsv', sep='\t')
/var/folders/q8/5rgt92sx7b74krhrt6l4_7hw0000gn/T/ipykernel_36662/2086606334.py:2: DtypeWarning: Columns (47,48,52,88,89,91,92,93,94,95,96,97,98,99,100,101,102,103,105,106,107,108) have mixed types. Specify dtype option on import or set low_memory=False.
  HWD_v4 = pd.read_csv('../data/figures/v4_df_GT_BH_allpop.tsv', sep='\t')


## Generate tag files (these cells have already been ran)

In [28]:
# import os
# os.makedirs('EnrichmentAnalysisFiles', exist_ok=True)

# # Generate HWD/HWE tag files — run this before any enrichment analysis
# HWD_all = pd.read_csv('../data/figures/v4_df_GT_BH_allpop.tsv', sep='	')
# all_all = pd.read_csv('../data/figures/v4_df_GT_allpop.tsv', sep='	')
# HWD_all = HWD_all.drop_duplicates(subset=['vid_v4pop'])
# all_all = all_all.drop_duplicates(subset=['vid_v4pop'])

# # Per population
# for pop in ['afr', 'amr', 'nfe', 'sas', 'fin', 'asj', 'eas']:
#     _hwd = HWD_all[HWD_all['pop'] == pop]
#     _all = all_all[all_all['pop'] == pop]
#     _hwd.to_csv(f'../output/EnrichmentAnalysisFiles/HWD_tags_{pop}.tsv', sep='	', index=False)
#     _all.to_csv(f'../output/EnrichmentAnalysisFiles/allVar_v4_{pop}.tsv', sep='	', index=False)
#     split_significant_variants(f'../output/EnrichmentAnalysisFiles/allVar_v4_{pop}.tsv', f'../output/EnrichmentAnalysisFiles/HWD_tags_{pop}.tsv', out_notsign=f'../output/EnrichmentAnalysisFiles/HWE_tags_{pop}.tsv')
#     print(f'{pop}: {len(_hwd)} HWD variants written.')

# # All populations combined
# HWD_all.to_csv('../output/EnrichmentAnalysisFiles/HWD_tags_allPop.tsv', sep='	', index=False)
# all_all.to_csv('../output/EnrichmentAnalysisFiles/allVar_v4_allPop.tsv', sep='	', index=False)
# split_significant_variants('../output/EnrichmentAnalysisFiles/allVar_v4_allPop.tsv', '../output/EnrichmentAnalysisFiles/HWD_tags_allPop.tsv', out_notsign='../output/EnrichmentAnalysisFiles/HWE_tags_allPop.tsv')
# print(f'allPop: {len(HWD_all)} HWD variants written.')

# # No AMR
# _hwd = HWD_all[HWD_all['pop'] != 'amr']
# _all = all_all[all_all['pop'] != 'amr']
# _hwd.to_csv('../output/EnrichmentAnalysisFiles/HWD_tags_noAMR.tsv', sep='	', index=False)
# _all.to_csv('../output/EnrichmentAnalysisFiles/allVar_v4_noAMR.tsv', sep='	', index=False)
# split_significant_variants('../output/EnrichmentAnalysisFiles/allVar_v4_noAMR.tsv', '../output/EnrichmentAnalysisFiles/HWD_tags_noAMR.tsv', out_notsign='../output/EnrichmentAnalysisFiles/HWE_tags_noAMR.tsv')
# print(f'noAMR: {len(_hwd)} HWD variants written.')

# # No AMR, no SAS
# _hwd = HWD_all[(HWD_all['pop'] != 'amr') & (HWD_all['pop'] != 'sas')]
# _all = all_all[(all_all['pop'] != 'amr') & (all_all['pop'] != 'sas')]
# _hwd.to_csv('../output/EnrichmentAnalysisFiles/HWD_tags_noAMRnoSAS.tsv', sep='	', index=False)
# _all.to_csv('../output/EnrichmentAnalysisFiles/allVar_v4_noAMRnoSAS.tsv', sep='	', index=False)
# split_significant_variants('../output/EnrichmentAnalysisFiles/allVar_v4_noAMRnoSAS.tsv', '../output/EnrichmentAnalysisFiles/HWD_tags_noAMRnoSAS.tsv', out_notsign='../output/EnrichmentAnalysisFiles/HWE_tags_noAMRnoSAS.tsv')
# print(f'noAMRnoSAS: {len(_hwd)} HWD variants written.')


# pLOF, syn, reg

In [ ]:
def fisherTable_pLOF_SynReg(var_HWE, var_HWD):

    features = {
        'pLOF': 'frameshift_variant|stop_gained|splice_donor_variant|splice_acceptor_variant|start_lost',
        'Syn':  'synonymous_variant',
        'Reg':  'regulatory_region_variant',
    }

    results = {}

    for prefix, pattern in features.items():
        print(f"\n===== {prefix} =====")

        HWD_in  = var_HWD['consequence'].str.contains(pattern).sum()
        HWD_out = len(var_HWD) - HWD_in
        HWE_in  = var_HWE['consequence'].str.contains(pattern).sum()
        HWE_out = len(var_HWE) - HWE_in

        contingency_table = np.array([
            [HWD_in, HWD_out],
            [HWE_in, HWE_out]
        ])

        OR, CI_low, CI_high, P = fisherTest(contingency_table)
        results[prefix] = (OR, CI_low, CI_high, P)

    return results


def sampleEnrich_pLOF_SynReg(HWD, HWE, timesSize):

    print('-------------------------------------pLOF / Synonymous / Regulatory----------------------------------------------------')

    spd_HWE = subPanda_popSpe(HWD, HWE, timesSize)
    return fisherTable_pLOF_SynReg(spd_HWE, HWD)


In [ ]:
import pandas as pd

group_display = {
    'afr':        'AFR',
    'amr':        'AMR',
    'nfe':        'NFE',
    'sas':        'SAS',
    'fin':        'FIN',
    'asj':        'ASJ',
    'eas':        'EAS',
    'allPop':     'Comb',
    'noAMR':      'Comb (no AMR)',
    'noAMRnoSAS': 'Comb (no AMR, no SAS)',
}

groups = [
    ('afr',        '../output/EnrichmentAnalysisFiles/HWD_tags_afr.tsv',        '../output/EnrichmentAnalysisFiles/HWE_tags_afr.tsv'),
    ('amr',        '../output/EnrichmentAnalysisFiles/HWD_tags_amr.tsv',        '../output/EnrichmentAnalysisFiles/HWE_tags_amr.tsv'),
    ('nfe',        '../output/EnrichmentAnalysisFiles/HWD_tags_nfe.tsv',        '../output/EnrichmentAnalysisFiles/HWE_tags_nfe.tsv'),
    ('sas',        '../output/EnrichmentAnalysisFiles/HWD_tags_sas.tsv',        '../output/EnrichmentAnalysisFiles/HWE_tags_sas.tsv'),
    ('fin',        '../output/EnrichmentAnalysisFiles/HWD_tags_fin.tsv',        '../output/EnrichmentAnalysisFiles/HWE_tags_fin.tsv'),
    ('asj',        '../output/EnrichmentAnalysisFiles/HWD_tags_asj.tsv',        '../output/EnrichmentAnalysisFiles/HWE_tags_asj.tsv'),
    ('eas',        '../output/EnrichmentAnalysisFiles/HWD_tags_eas.tsv',        '../output/EnrichmentAnalysisFiles/HWE_tags_eas.tsv'),
    ('allPop',     '../output/EnrichmentAnalysisFiles/HWD_tags_allPop.tsv',     '../output/EnrichmentAnalysisFiles/HWE_tags_allPop.tsv'),
    ('noAMR',      '../output/EnrichmentAnalysisFiles/HWD_tags_noAMR.tsv',      '../output/EnrichmentAnalysisFiles/HWE_tags_noAMR.tsv'),
    ('noAMRnoSAS', '../output/EnrichmentAnalysisFiles/HWD_tags_noAMRnoSAS.tsv', '../output/EnrichmentAnalysisFiles/HWE_tags_noAMRnoSAS.tsv'),
]

rows = []

for label, sign_file, notSign_file in groups:
    print(f"\n{'='*60}")
    print(f"Group: {label}")
    print(f"{'='*60}")
    try:
        var_sign    = pd.read_csv(sign_file, sep='\t')
        var_notSign = pd.read_csv(notSign_file, sep='\t')
        print(len(var_sign))
        print(len(var_notSign))

        res = sampleEnrich_pLOF_SynReg(var_sign, var_notSign, 100)

        rows.append({
            'Group':        group_display[label],
            'pLOF_OR':      res['pLOF'][0], 'pLOF_CI_low':  res['pLOF'][1],
            'pLOF_CI_high': res['pLOF'][2], 'pLOF_P':       res['pLOF'][3],
            'Syn_OR':       res['Syn'][0],  'Syn_CI_low':   res['Syn'][1],
            'Syn_CI_high':  res['Syn'][2],  'Syn_P':        res['Syn'][3],
            'Reg_OR':       res['Reg'][0],  'Reg_CI_low':   res['Reg'][1],
            'Reg_CI_high':  res['Reg'][2],  'Reg_P':        res['Reg'][3],
        })

    except FileNotFoundError as e:
        print(f'Missing file: {e.filename}')

df_results = pd.DataFrame(rows)
tsv_out = '../output/EnrichmentAnalysisFiles/enrichment_results.tsv'
df_results.to_csv(tsv_out, sep='\t', index=False)
print(f'\nResults saved to {tsv_out}')


# Figure 2l

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_enrich = pd.read_csv('../output/EnrichmentAnalysisFiles/enrichment_results.tsv', sep='\t')

features    = ['Synonymous',  'Regulatory region', 'pLOF']
or_cols     = ['Syn_OR',      'Reg_OR',            'pLOF_OR']
ci_low_cols = ['Syn_CI_low',  'Reg_CI_low',        'pLOF_CI_low']
ci_hi_cols  = ['Syn_CI_high', 'Reg_CI_high',       'pLOF_CI_high']
p_cols      = ['Syn_P',       'Reg_P',             'pLOF_P']
feat_colors = {
    'Synonymous':        '#E67E22',
    'Regulatory region': '#2ECC71',
    'pLOF':              '#9B59B6',
}
groups      = df_enrich['Group'].tolist()
n_groups    = len(groups)
y_positions = list(range(n_groups))

fig, axes = plt.subplots(1, 3, figsize=(7, 5), dpi=300)
fig.subplots_adjust(wspace=0.3)

for ci, (ax, feat, or_col, cil_col, cih_col, p_col) in enumerate(
        zip(axes, features, or_cols, ci_low_cols, ci_hi_cols, p_cols)):
    col = feat_colors[feat]

    for gi, grp in enumerate(groups):
        ax.axhspan(gi - 0.5, gi + 0.5,
                   color='#f5f5f5' if gi % 2 == 0 else 'white', zorder=0)
        row    = df_enrich.loc[df_enrich['Group'] == grp]
        or_val = row[or_col].values[0]
        ci_lo  = row[cil_col].values[0]
        ci_hi  = row[cih_col].values[0]
        p_val  = row[p_col].values[0]
        alpha  = 1.0 if p_val < 0.05 else 0.25

        ax.errorbar(or_val, gi,
                    xerr=[[or_val - ci_lo], [ci_hi - or_val]],
                    fmt='x', color=col, alpha=alpha,
                    markersize=8, markeredgewidth=1.5,
                    elinewidth=0.5, capsize=1.5, capthick=1.0,
                    zorder=3)
        if p_val < 0.05:
            ax.text(ci_hi, gi + 0.3, f'{or_val:.2f}',
                    ha='left', va='bottom', fontsize=7, color=col)

    if ci == 0:
        ax.set_xlim(0, 1)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['0', '1'])
    else:
        ax.set_xscale('log')
        ax.set_xlim(0.5, 20)
        ax.set_xticks([1, 10])
        ax.set_xticklabels(['1', '10'])
    ax.axvline(1, color='#333333', lw=0.9, linestyle='--', zorder=2)

    ax.set_yticks(y_positions)
    display_labels = [g.replace('Comb (no AMR, no SAS)', 'Comb\n(no AMR, no SAS)')
                       .replace('Comb (no AMR)', 'Comb\n(no AMR)') for g in groups]
    ax.set_yticklabels(display_labels if ci == 0 else [], fontsize=9)
    ax.invert_yaxis()

    ax.set_title(feat, fontsize=10, fontweight='bold', color=col, pad=6)
    ax.set_xlabel('OR', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(ci == 0)
    ax.xaxis.grid(True, linestyle=':', linewidth=0.5, color='#cccccc', zorder=1)
    ax.tick_params(labelsize=9)

fig.suptitle('Enrichment analysis - gnomAD v4 CHWD variants  (faded = p >= 0.05)',
             fontsize=10, y=1.02)
plt.tight_layout()
# plt.savefig('Figures_2l_TableS2/enrichment_v4_forest_log_3panel_CI.pdf', bbox_inches='tight', dpi=300)
# plt.savefig('Figures_2l_TableS2/enrichment_v4_forest_log_3panel_CI.png', bbox_inches='tight', dpi=300)
plt.show()
# print('Saved: enrichment_v4_forest_log_3panel_CI.pdf / .png')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df = pd.read_csv('../output/EnrichmentAnalysisFiles/enrichment_results.tsv', sep='\t')

features = [
    ('Synonymous',        'Syn'),
    ('Regulatory region', 'Reg'),
    ('pLOF',              'pLOF'),
]

def fmt_cell(or_val, ci_lo, ci_hi, p_val):
    if or_val == 0 or np.isnan(or_val):
        return '—', False
    ci_str = f'{or_val:.2f} ({ci_lo:.2f}–{ci_hi:.2f})'
    p_str  = f'{p_val:.2e}' if p_val < 0.001 else f'{p_val:.3f}'
    return f'{ci_str}\np = {p_str}', p_val < 0.05

groups  = df['Group'].tolist()
n_rows  = len(groups)
n_cols  = len(features)

cell_text   = []
cell_colors = []

for grp in groups:
    row_text   = []
    row_colors = []
    r = df[df['Group'] == grp].iloc[0]
    for feat_label, prefix in features:
        text, sig = fmt_cell(r[f'{prefix}_OR'], r[f'{prefix}_CI_low'],
                             r[f'{prefix}_CI_high'], r[f'{prefix}_P'])
        row_text.append(text)
        row_colors.append('#d4edda' if sig else '#ffffff')
    cell_text.append(row_text)
    cell_colors.append(row_colors)

fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
ax.axis('off')

tbl = ax.table(
    cellText=cell_text,
    rowLabels=groups,
    colLabels=[f[0] for f in features],
    cellColours=cell_colors,
    cellLoc='center',
    loc='center',
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 3.2)

for j in range(n_cols):
    tbl[0, j].set_facecolor('#2c3e50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')

for i in range(1, n_rows + 1):
    tbl[i, -1].set_facecolor('#ecf0f1')
    tbl[i, -1].set_text_props(fontweight='bold')

for i in range(n_rows):
    for j, (_, prefix) in enumerate(features):
        r = df.iloc[i]
        _, sig = fmt_cell(r[f'{prefix}_OR'], r[f'{prefix}_CI_low'],
                          r[f'{prefix}_CI_high'], r[f'{prefix}_P'])
        if sig:
            tbl[i + 1, j].set_text_props(fontweight='bold')

fig.suptitle('Enrichment analysis for CHWD variants in gnomAD',
             fontsize=14, fontweight='bold', y=1)
plt.tight_layout()
# plt.savefig('Figures_2l_TableS2/enrichment_v4_table.pdf', bbox_inches='tight', dpi=300)
# plt.savefig('Figures_2l_TableS2/enrichment_v4_table.png', bbox_inches='tight', dpi=300)
plt.show()
# print('Saved: enrichment_v4_table.pdf / .png')


# Exons vs flanking regions variants

In [ ]:
# Extract all unique consequences from allVar_v4
import re

all_csq = set()
for entry in allVar_v4['consequence'].dropna():
    tokens = re.split(r'[&;]', str(entry))
    for t in tokens:
        t = t.strip()
        if t:
            all_csq.add(t)

for csq in sorted(all_csq):
    print(csq)


In [ ]:
import re

EXON_CONSEQUENCES = {
    'coding_sequence_variant',
    'frameshift_variant',
    'incomplete_terminal_codon_variant',
    'inframe_deletion',
    'inframe_insertion',
    'missense_variant',
    'splice_acceptor_variant',
    'splice_donor_variant',
    'stop_gained',
    'stop_lost',
    'stop_retained_variant',
    'synonymous_variant',
}

# Deduplicate allVar_v4 — one row per unique variant (ignore population replicates)
allVar_v4_unique = allVar_v4.drop_duplicates(['chromo', 'position', 'ref', 'alt']).reset_index(drop=True)
n_total = len(allVar_v4_unique)
print(f'Total unique variants: {n_total:,}')

n_exon     = 0
n_flanking = 0
n_both     = 0

for csq_str in allVar_v4_unique['consequence'].dropna():
    tokens = {t.strip() for t in re.split(r'[&;]', str(csq_str)) if t.strip()}
    has_exon     = bool(tokens & EXON_CONSEQUENCES)
    has_flanking = bool(tokens - EXON_CONSEQUENCES)
    if has_exon:
        n_exon += 1
    if has_flanking:
        n_flanking += 1
    if has_exon and has_flanking:
        n_both += 1

print(f'\nExon only (at least one exonic consequence):')
print(f'  {n_exon:>8,} / {n_total:,}  ({n_exon/n_total*100:.1f}%)')
print(f'\nFlanking (at least one non-exonic consequence):')
print(f'  {n_flanking:>8,} / {n_total:,}  ({n_flanking/n_total*100:.1f}%)')
print(f'\nBoth (exonic AND non-exonic consequences):')
print(f'  {n_both:>8,} / {n_total:,}  ({n_both/n_total*100:.1f}%)')
print(f'\nExon only (no flanking):')
n_exon_only = n_exon - n_both
print(f'  {n_exon_only:>8,} / {n_total:,}  ({n_exon_only/n_total*100:.1f}%)')
print(f'\nFlanking only (no exon):')
n_flanking_only = n_flanking - n_both
print(f'  {n_flanking_only:>8,} / {n_total:,}  ({n_flanking_only/n_total*100:.1f}%)')
